# ROM-FlameBench: a simple walkthrough

This notebook follows one complete **POD → ARX** experiment:

1. Prepare and load the data.
2. Compress the fields into a small latent state.
3. Learn how that state evolves under the inlet forcing.
4. Evaluate recursive forecasts on unseen trajectories.

Run the cells in order, from the repository root, using the project's Python environment.
POD fitting and full test rollouts use the real dataset and can take time: this is a
walkthrough of an experiment, not a tiny synthetic demo.

**Confirmed:** snapshot zero is the initial steady state.  
**TODO:** add physical cell volumes. Until then, we evaluate fields only; integrated
heat release and its gain/phase metrics are disabled.

## 1. Data preparation

Raw archives contain `data` with shape **(cell, field, time)**. Preparation writes
float32 `.npy` files with shape **(time, field, cell)**. These files can be read from
disk in small batches. Existing prepared files are reused.

The supplied `phi` files contain the dimensionless forcing: $U(t)=U_{base}\phi(t)$.
The metadata defines filenames, field order, units and the sampling interval.

In [1]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from DataProcessing.prepare import prepare
from DataProcessing.metadata import load_metadata
from DataProcessing.Dataset import TrainingDataset, TestDataset
from utils import seed_everything, write_json
from Experiments.paths import new_run_name, run_directory

seed = 42
seed_everything(seed)
torch.set_num_threads(4)

metadata_path = Path("Data/metadata.json")
raw_directory = Path("Data/Raw")
run_settings = {
    "output": "Experiments/Results",
    "seed": seed,
    "compressor": {"name": "pod"},
    "model": {"name": "arx"},
}
run_settings["run_name"] = new_run_name(run_settings)
output = run_directory(run_settings)
print("Run directory:", output)

Nx = 9  # Past snapshots, plus the current snapshot: 10 in total.
Ni = 4  # Past inputs, plus current and next forcing: 6 in total.
rank = 16
batch_size = 64
validation_fraction = 0.3

# .env can enable online W&B logging for the FireMark team.
logging_config = {"logging": {"wandb": {"mode": "offline", "entity": "FireMark"}}}
output.mkdir(parents=True, exist_ok=True)

In [2]:
prepare(metadata_path, raw_directory)
metadata = load_metadata(metadata_path)

print("Fields:", metadata["fields"])
print("Sampling interval:", metadata["dt"], "seconds")
for case in metadata["cases"]:
    print(case["split"], case["name"])

Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Training/sineSweep_f1_f80_A02.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Training/phi_sineSweep_f1_f80_A02.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Training/sineSweep_f1_f80_A04.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Training/phi_sineSweep_f1_f80_A04.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Test/sine_f10_A03.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Test/phi_sine_f10_A03.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Test/sine_f10_A05.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Test/phi_sine_f10_A05.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Test/sine_f40_A03.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Test/phi_sine_f40_A03.npy
Keeping /Users/carlofab/PyCharmMiscProject/FlameBench/Data/Test/sine_f40_A05.npy
Keeping /Users/carlofab/PyCharmMiscProjec

The first **70% of each training sweep** is used for training; the last **30%** is
validation. Windows never cross a split or a simulation boundary. Test simulations
are kept separate.

A sample contains `Nx + 1` fields, `Ni + 2` inputs $[\phi(t-N_i),\ldots,\phi(t+1)]$, and the
next field $y(t+1)$. **Nx and Ni are independent** and count observations strictly
before the current time. Windows need `max(Nx, Ni) + 1` context samples within each
split. The `Dataset` defines samples; a `DataLoader` groups them into batches.

In [3]:
training = TrainingDataset(metadata, Nx=Nx, Ni=Ni, validation_fraction=validation_fraction)
validation = TrainingDataset(metadata, Nx=Nx, Ni=Ni, validation_fraction=validation_fraction, partition="validation")
testing = TestDataset(metadata, Nx=Nx, Ni=Ni)

sample = training[0]
print("Training windows:", len(training))
print("Validation windows:", len(validation))
print("Test windows:", len(testing))
print("History:", tuple(sample["history"].shape))  # (Nx + 1, fields, cells)
print("Forcing:", sample["forcing"].tolist())     # phi(t-Ni), ..., phi(t+1)
print("Target:", tuple(sample["target"].shape))    # (fields, cells)

Training windows: 5580
Validation windows: 2382
Test windows: 11946
History: (10, 11, 21334)
Forcing: [1.003158688545227, 1.003794550895691, 1.0044317245483398, 1.0050703287124634, 1.005710244178772, 1.0063514709472656]
Target: (11, 21334)


## 2. Compressor

The fields have different units and scales. First, we estimate a mean and standard
deviation for each field **using training data only**. Scaling keeps one field from
dominating simply because its numerical values are larger.

POD then maps each scaled snapshot to `rank` coefficients. Here POD uses incremental
SVD, so it processes batches instead of collecting the full dataset in RAM. It is a
centered Euclidean approximation to batch POD; it does not use cell-volume weights.

In [4]:
from DataProcessing.scaling import FeatureScaler
from Baselines.OrderReduction.Linear.POD import POD

scaler = FeatureScaler()
scaler.fit(training.snapshot_batches(batch_size))

compressor = POD(rank=rank, batch_size=batch_size)
compressor.fit(training, scaler)
print("POD coefficients per snapshot:", compressor.rank)

POD coefficients per snapshot: 16


`encode` converts a scaled field to coefficients; `decode` reconstructs the scaled
field. Inverse scaling returns the original physical units. This reconstruction check
measures compression alone, before any forecasting.

In [5]:
snapshots = next(validation.snapshot_batches(2))
scaled = scaler.transform(snapshots)
latent = compressor.encode(scaled)
reconstructed = scaler.inverse(compressor.decode(latent))

print("Field batch:", snapshots.shape)
print("Latent batch:", latent.shape)
print("Reconstructed batch:", reconstructed.shape)
print("Scaled reconstruction MSE:", np.mean((compressor.decode(latent) - scaled) ** 2))

Field batch: (2, 11, 21334)
Latent batch: (2, 16)
Reconstructed batch: (2, 11, 21334)
Scaled reconstruction MSE: 0.017415479


## 3. Forecast

We first encode the training and validation trajectories into small, disk-backed
latent arrays. This avoids re-encoding large fields at every training step.

**ARX** is a linear regression from a history of POD coefficients and the prescribed
forcing history (including the next prescribed input) to the next coefficients:

$$\widehat z_{t+1} = f(z_{t-N_x},\ldots,z_t,\phi_{t-N_i},\ldots,\phi_{t+1}).$$

`alpha` controls ridge regularization. ARX uses a direct fit, so it has no epochs.
The same latent datasets can also train a GRU, LSTM or Transformer. In those models,
compressed snapshots **and forcing enter the recurrent/attention block together**,
aligned by time. Availability flags handle different `Nx` and `Ni`; the final time
contains known next forcing but no future state. The compressor processes fields only.

In [6]:
from DataProcessing.latent import LatentDataset
from Baselines.Forecast.Classical.ARX import ARX

train_latent = LatentDataset(training, compressor, scaler, output / "latent" / "training")
validation_latent = LatentDataset(validation, compressor, scaler, output / "latent" / "validation")

train_loader = DataLoader(train_latent, batch_size=batch_size, shuffle=False)
validation_loader = DataLoader(validation_latent, batch_size=batch_size, shuffle=False)

model = ARX(alpha=1e-4)
model.fit(train_loader, validation_loader)

Validation checks a **recursive rollout**, not just one-step prediction. It starts
after enough context for both histories inside the validation segment and then feeds predictions
back into the model. The test set does not participate in this step.

This latent MSE is suitable for choosing forecasters with the same compressor. To
compare different compressors or ranks, use a shared field-space validation metric.

In [7]:
from Experiments.run import validation_rollout, dump

validation_error = validation_rollout(model, validation_latent)
if not np.isfinite(validation_error):
    raise RuntimeError("Validation rollout diverged. Review the model before testing.")

print("Recursive validation latent MSE:", validation_error)

# Save the fitted objects so they can be reused without fitting again.
dump(output / "preprocessing.pkl", (scaler, compressor))
dump(output / "model.pkl", model)

Recursive validation latent MSE: 447.42719144036255


## 4. Evaluation

Each test rollout starts from **snapshot zero**, repeated to fill the model's history.
No later ground-truth field is supplied to the model. Forcing before time zero is
padded with **1**; from time zero onward, we use the supplied forcing. In step cases,
`phi[0]` is already perturbed. No forcing after the next prediction time is used.

The evaluator repeats this sequence until the end of each test simulation:

1. Predict the next latent state from the latent history and $[\phi_{t-N_i},\ldots,\phi_{t+1}]$.
2. Decode it and return to physical units.
3. Compare with the reference field and update the history with the prediction.

For each field, NRMSE is RMSE divided by the ground-truth standard deviation, pooling
all evaluated cells and times in that case. The reported mean is the average of the
11 field scores. Snapshot zero is excluded because it was given to the model.

**TODO — cell volumes:** `heat_release=False` explicitly skips integrated $Q(t)$ and
its gain/phase metrics. Local `mix:Q` is still included in the field scores.

In [8]:
from Experiments.evaluation import evaluate

results = evaluate(
    model=model,
    dataset=testing,
    compressor=compressor,
    scaler=scaler,
    directory=output,
    initialization="steady",
    heat_release=False,  # TODO: supply physical cell volumes before enabling Q(t).
    save_predictions=False,
)

In [9]:
for name, metrics in results.items():
    print(f"\n{name}")
    print("Mean NRMSE:", metrics["mean_nrmse"])
    for field, error in metrics["field_nrmse"].items():
        print(f"  {field}: {error}")
    print("Seconds per forecast step:", metrics["seconds_per_step"])


sine_f10_A03
Mean NRMSE: 0.1635915970744692
  p: 0.17437290009397224
  U1: 0.3365726978551102
  U3: 0.07484458077971011
  rho: 0.06352950581773861
  T: 0.08233712387145785
  mix:Q: 0.3568034225248519
  CH4: 0.04982386668330233
  O2: 0.06868253265165285
  H2O: 0.06993561902288836
  CO2: 0.10799011586933464
  OH: 0.4146152026491421
Seconds per forecast step: 0.0003704081539690378

sine_f10_A05
Mean NRMSE: 0.29539011810742744
  p: 0.19401072826230506
  U1: 0.528465758821642
  U3: 0.15223427073906085
  rho: 0.13800506728640521
  T: 0.19907903883644676
  mix:Q: 0.590618515918747
  CH4: 0.11804088161302073
  O2: 0.16339613635164207
  H2O: 0.17237833903262326
  CO2: 0.24970265288359259
  OH: 0.743359909436216
Seconds per forecast step: 0.00035215386244817637

sine_f40_A03
Mean NRMSE: 0.13494358796389871
  p: 0.1633752041085414
  U1: 0.22020439472703912
  U3: 0.038498336106138245
  rho: 0.04781153337336835
  T: 0.05490637773804778
  mix:Q: 0.4913085895883002
  CH4: 0.058897106293741626
  O2: 

Finally, save the settings and send the scalar results to **TensorBoard and W&B**.
W&B defaults to offline here; the project's `.env` can set `WANDB_MODE=online` and
provide the API key for `FireMark`. Credentials are not saved with these settings.

The detailed metrics are already in `metrics.json` inside the seed folder. TensorBoard can read
the run with `tensorboard --logdir Experiments/Results`.
Running the setup cell creates a new `pod_arx_<timestamp>/seed_<seed>` directory.
Re-running later cells uses that same seed directory. CLI multi-seed runs share one
timestamp across seeds with `--seeds 0 1 2`.

In [10]:
from Experiments.logging import ExperimentLogger

settings = {
    "Nx": Nx,
    "Ni": Ni,
    "rank": rank,
    "batch_size": batch_size,
    "validation_fraction": validation_fraction,
    "compressor": "pod",
    "model": "arx",
    "alpha": 1e-4,
    "seed": seed,
    "run_name": run_settings["run_name"],
    "heat_release": False,
    **logging_config,
}
write_json(output / "notebook_settings.json", settings)

logger = ExperimentLogger(output, settings)
try:
    logger.log({"validation/rollout_latent_mse": validation_error}, step=0)
    for name, metrics in results.items():
        values = {f"test/{name}/mean_nrmse": metrics["mean_nrmse"]}
        for field, error in metrics["field_nrmse"].items():
            values[f"test/{name}/nrmse/{field}"] = error
        logger.log(values, step=1)
finally:
    logger.close()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: carlo-fabrizio000 (FireMark) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 9756e8a5
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /Users/carlofab/PyCharmMiscProject/FlameBench/Experiments/Results/notebook_pod_arx/wandb/run-20260920_170759-9756e8a5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run notebook_pod_arx
wandb: ⭐️ View project at https://wandb.ai/FireMark/FireMark
wandb: 🚀 View run at https://wandb.ai/FireMark/FireMark/runs/9756e8a5
wandb: updating run metadata; uploading data
wandb: uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:                        epoch ▁██████
wandb: test/sine_f10_A03/mean_nrmse ▁
wandb:  test/sine_f10_A03/nrmse/CH4 ▁
wandb:  test/sine_f10_A03/nrmse/CO2 ▁
wandb:  test/sine_f10_A03/nrmse/H2O ▁
wandb:   test/sine_f10_A03/nrmse